In [1]:
import numpy as np

# Читаем матрицу из файла в numpy массив
matrix = np.loadtxt('config/a_max_glin.in')

print(matrix)

[[  1.5   35.     2.65  60.     1.  ]
 [  7.    40.     2.65  50.     1.  ]
 [ 10.    20.     2.65  -1.8    1.  ]
 [  0.     0.     2.65  -2.     1.  ]
 [  0.     0.     1.   100.     1.  ]]


In [5]:
from pathlib import Path
import lasio as ls

# Путь к LAS файлу (используйте свой путь при необходимости)
las_path = Path("data/las/skv621.las")

# Загружаем данные из LAS в том же стиле, что и в mkm_core.py
lasdata = ls.read(las_path)
keys = list(lasdata.keys())

# Экземпляр кода из функции load_mkm_from_las
def infer_property_mnemonics(
    las_keys,
    prop_mnems=None,
):
    STANDARD_PROP_TRIPLE = ("POTA", "THOR", "RHOB")
    FOURTH_PROP_CANDIDATES = (
        "TRNP",
        "WNKT",
        "NPHI",
        "TNPH",
        "NPOR",
        "CACO",
        "GR",
        "CNP",
    )
    keyset = set(las_keys)
    if prop_mnems is not None:
        props = tuple(prop_mnems)
        if len(props) != 4:
            raise ValueError(
                "Нужно ровно 4 мнемоники кривых-свойств (после LITO идут 4 столбца + единицы → 5×5)."
            )
        missing = [m for m in props if m not in keyset]
        if missing:
            raise ValueError(
                f"В LAS нет кривых: {missing}. Доступные мнемоники: {sorted(keyset)}"
            )
        return props

    missing_std = [m for m in STANDARD_PROP_TRIPLE if m not in keyset]
    if missing_std:
        raise ValueError(
            f"Автовыбор свойств: ожидаются {STANDARD_PROP_TRIPLE}. Не хватает: {missing_std}. "
            f"Доступно: {sorted(keyset)}. Задайте явно: --props M1 M2 M3 M4"
        )
    for cand in FOURTH_PROP_CANDIDATES:
        if cand in keyset:
            return (*STANDARD_PROP_TRIPLE, cand)
    raise ValueError(
        f"Не найдена четвёртая кривая из списка {FOURTH_PROP_CANDIDATES}. "
        "Укажите явно: --props M1 M2 M3 M4"
    )

depth_mnem = "DEPT"
litho_mnem = "LITO"
prop_mnems = None

keyset = set(keys)
for req, label in ((depth_mnem, "глубины"), (litho_mnem, "литологии")):
    if req not in keyset:
        raise ValueError(
            f"Кривая {req!r} ({label}) отсутствует в LAS. Доступно: {sorted(keyset)}"
        )

props = infer_property_mnemonics(keys, prop_mnems)
curve_order = [depth_mnem, litho_mnem, *props]
data = lasdata.stack_curves(curve_order, sort_curves=False)

litho_raw = np.asarray(data[:, 1], dtype=float).copy()

data = np.c_[data, np.ones(data.shape[0])]
data[data[:, 1] != 1, 1] = 2

is_coll = data[:, 1] == 1
is_glin = data[:, 1] == 2

coll_prop = data[is_coll][:, 2:]
glin_prop = data[is_glin][:, 2:]

print("shape data", data.shape)
print("litho_raw", litho_raw[:10])
print("coll_prop.shape", coll_prop.shape)
print("glin_prop.shape", glin_prop.shape)

# Вывести строку с одной глубины (например, первой)
print("\nОдна строка данных с глубины:")
print("Заголовки:", curve_order + ["ONES"])
print("Данные:", data[0])

shape data (400, 7)
litho_raw [2. 4. 4. 4. 4. 4. 4. 4. 4. 4.]
coll_prop.shape (104, 5)
glin_prop.shape (296, 5)

Одна строка данных с глубины:
Заголовки: ['DEPT', 'LITO', 'POTA', 'THOR', 'RHOB', 'WNKT', 'ONES']
Данные: [2.92000e+03 2.00000e+00 9.35280e+00 4.99784e+01 2.28916e+00 3.99351e+01
 1.00000e+00]


In [11]:
las_example = data[0][2:]
las_example

array([ 9.3528 , 49.9784 ,  2.28916, 39.9351 ,  1.     ])

In [7]:
inv_m_coll = np.linalg.inv(matrix)

print(inv_m_coll)

[[-2.13664596e+00  2.55279503e+00 -1.36645963e+00  9.50310559e-01
  -4.99600361e-16]
 [ 1.07370600e+00 -1.28302277e+00  7.37060041e-01 -5.27743271e-01
   2.49800181e-16]
 [-3.32768681e+01  4.09561453e+01 -2.36777715e+01  1.66045549e+01
  -6.06060606e-01]
 [-5.38302277e-01  6.62525880e-01 -3.83022774e-01  2.58799172e-01
  -1.30024996e-16]
 [ 8.71070958e+01 -1.07208733e+02  6.19800489e+01 -4.24844720e+01
   1.60606061e+00]]


In [9]:
mkm = las_example @ inv_m_coll
print(mkm)


[ 23.11235103 -27.2431705   16.5386467  -11.62651814   0.21869091]


In [10]:
mkm.sum()

np.float64(1.000000000000019)

In [12]:
matrix

array([[  1.5 ,  35.  ,   2.65,  60.  ,   1.  ],
       [  7.  ,  40.  ,   2.65,  50.  ,   1.  ],
       [ 10.  ,  20.  ,   2.65,  -1.8 ,   1.  ],
       [  0.  ,   0.  ,   2.65,  -2.  ,   1.  ],
       [  0.  ,   0.  ,   1.  , 100.  ,   1.  ]])

In [13]:
# Нужно в матрице matrix первый и второй столбец попарно умножить на значения из третьего столбца
# Предполагается, что matrix имеет форму (n, 3) или больше столбцов
matrix[:, 0] = matrix[:, 0] * matrix[:, 2]
matrix[:, 1] = matrix[:, 1] * matrix[:, 2]
matrix

array([[  3.975,  92.75 ,   2.65 ,  60.   ,   1.   ],
       [ 18.55 , 106.   ,   2.65 ,  50.   ,   1.   ],
       [ 26.5  ,  53.   ,   2.65 ,  -1.8  ,   1.   ],
       [  0.   ,   0.   ,   2.65 ,  -2.   ,   1.   ],
       [  0.   ,   0.   ,   1.   , 100.   ,   1.   ]])

In [14]:
las_example

array([ 9.3528 , 49.9784 ,  2.28916, 39.9351 ,  1.     ])

In [15]:
las_example[0] = las_example[0] * las_example[2]
las_example[1] = las_example[1] * las_example[2]
las_example


array([ 21.41005565, 114.40855414,   2.28916   ,  39.9351    ,
         1.        ])

In [16]:
inv_m_coll = np.linalg.inv(matrix)
mkm = las_example @ inv_m_coll
mkm

array([ 18.52648507, -21.76280828,  13.26291967,  -9.24528736,
         0.21869091])

In [17]:
matrix[:, 0] = matrix[:, 0] / 1000
matrix[:, 1] = matrix[:, 1] / 1000
matrix

array([[ 3.975e-03,  9.275e-02,  2.650e+00,  6.000e+01,  1.000e+00],
       [ 1.855e-02,  1.060e-01,  2.650e+00,  5.000e+01,  1.000e+00],
       [ 2.650e-02,  5.300e-02,  2.650e+00, -1.800e+00,  1.000e+00],
       [ 0.000e+00,  0.000e+00,  2.650e+00, -2.000e+00,  1.000e+00],
       [ 0.000e+00,  0.000e+00,  1.000e+00,  1.000e+02,  1.000e+00]])

In [18]:
las_example[0] = las_example[0] /1000
las_example[1] = las_example[1] /1000
las_example

array([2.14100556e-02, 1.14408554e-01, 2.28916000e+00, 3.99351000e+01,
       1.00000000e+00])

In [21]:
inv_m_coll = np.linalg.inv(matrix)
mkm = las_example @ inv_m_coll
# mkm = inv_m_coll @ las_example
mkm

array([ 18.52648507, -21.76280828,  13.26291967,  -9.24528736,
         0.21869091])

In [ ]:
import numpy as np

# Явно задаём matrix и las_example, чтобы можно было менять значения вручную.
# matrix имеет форму (n, 3) или (n, 4|5...) в зависимости от задачи — демонстрация на (5, 4)
matrix = np.array([
    [1.5, 35.0, 2.65, 60.0,1],
    [7.0, 40.0, 2.65, 50.0,1],
    [10.0, 20.0, 2.65, -1.8,1],
    [0.0, 0.0, 2.65, -2.0,1],
    [0.0, 0.0, 1.0, 100.0,1],
], dtype=float)

# las_example должен быть одним вектором длины как количество столбцов в matrix (здесь 4)
las_example = np.array([9.3528, 49.9784, 2.28916, 39.9351, 1], dtype=float)

# Для удобства можно выводить
print("matrix:\n", matrix)
print("las_example:\n", las_example)

matrix:
 [[  1.5   35.     2.65  60.     1.  ]
 [  7.    40.     2.65  50.     1.  ]
 [ 10.    20.     2.65  -1.8    1.  ]
 [  0.     0.     2.65  -2.     1.  ]
 [  0.     0.     1.   100.     1.  ]]
las_example:
 [ 9.3528  49.9784   2.28916 39.9351   1.     ]


In [26]:
mkm = las_example @ np.linalg.inv(matrix)
mkm

array([ 23.11235103, -27.2431705 ,  16.5386467 , -11.62651814,
         0.21869091])

In [65]:
import numpy as np

# Явно задаём matrix и las_example, чтобы можно было менять значения вручную.
# matrix имеет форму (n, 3) или (n, 4|5...) в зависимости от задачи — демонстрация на (5, 4)
matrix = np.array([
    [0.75, 22.5, 2.65, 57, 1],
    [4, 25, 2.65, 38, 1],
    [6, 12.5, 2.65, -1.8, 1],
    [0.0, 0.0, 2.65, -2.0, 1],
    [0.0, 0.0, 1.0, 100.0, 1],
], dtype=float)

# las_example должен быть одним вектором длины как количество столбцов в matrix (здесь 4)
las_example = np.array([9.3528, 49.9784, 2.28916, 39.9351, 1], dtype=float)

las_example @ np.linalg.inv(matrix) 

array([-6.03017661,  9.40501337, -3.95743684,  1.36390917,  0.21869091])

In [62]:
glin_prop[:, 0] = glin_prop[:, 0] * glin_prop[:, 2]
glin_prop[:, 1] = glin_prop[:, 1] * glin_prop[:, 2]



In [63]:
matrix[:, 0] = matrix[:, 0] * matrix[:, 2]
matrix[:, 1] = matrix[:, 1] * matrix[:, 2]
matrix





array([[  3.975,  92.75 ,   2.65 ,  60.   ,   1.   ],
       [ 18.55 , 106.   ,   2.65 ,  50.   ,   1.   ],
       [ 26.5  ,  53.   ,   2.65 ,  -1.8  ,   1.   ],
       [  0.   ,   0.   ,   2.65 ,  -2.   ,   1.   ],
       [  0.   ,   0.   ,   1.   , 100.   ,   1.   ]])

In [66]:
glin_prop @ np.linalg.inv(matrix) 

array([[-1.66496327e+01,  2.51043574e+01, -1.10866916e+01,
         3.41327597e+00,  2.18690909e-01],
       [-1.61628472e+01,  2.43817285e+01, -1.07352840e+01,
         3.29873603e+00,  2.17666667e-01],
       [-1.60832883e+01,  2.42704455e+01, -1.06796034e+01,
         3.27567649e+00,  2.16769697e-01],
       ...,
       [ 9.00214339e-01, -8.72210390e-01,  1.22371160e+00,
        -2.35624636e-01, -1.60909091e-02],
       [ 1.05791809e+00, -1.08420862e+00,  1.32448027e+00,
        -2.75317008e-01, -2.28727273e-02],
       [ 1.26280942e+00, -1.38547206e+00,  1.41590635e+00,
        -3.05558863e-01,  1.23151515e-02]], shape=(296, 5))

In [67]:
matrix

array([[  0.75,  22.5 ,   2.65,  57.  ,   1.  ],
       [  4.  ,  25.  ,   2.65,  38.  ,   1.  ],
       [  6.  ,  12.5 ,   2.65,  -1.8 ,   1.  ],
       [  0.  ,   0.  ,   2.65,  -2.  ,   1.  ],
       [  0.  ,   0.  ,   1.  , 100.  ,   1.  ]])